In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


### Persisting the FAISS index and embeddings

Encoding ~20k descriptions and rebuilding the HNSW index takes a couple of minutes and gives
the exact same result every time it's re-run on the same data, so both are cached to disk on
the first run and reloaded on every subsequent one:

- **`data/game_features.parquet`** - the full `df` (metadata + `Description`) plus an
  `embedding` column, one vector per row, row-aligned with `df` automatically.
- **`data/faiss_index.bin`** - the FAISS `IndexHNSWFlat`, saved with `faiss.write_index()` /
  reloaded with `faiss.read_index()`.

These are the same cache files used in `8_learning_to_rank.ipynb`, so whichever notebook you
run first builds them and the other one just loads them - no need to encode twice.

If you change the embedding model, the source CSVs, or the preprocessing, delete these two
files (or point `FEATURES_PATH`/`FAISS_INDEX_PATH` at a new name) so they get rebuilt - the
cache is intentionally dumb and will keep serving stale data otherwise.

In [ ]:
DATA_DIR = '../data'
FEATURES_PATH = os.path.join(DATA_DIR, 'game_features.parquet')
FAISS_INDEX_PATH = os.path.join(DATA_DIR, 'faiss_index.bin')

USE_CACHE = os.path.exists(FEATURES_PATH) and os.path.exists(FAISS_INDEX_PATH)

if USE_CACHE:
    print("Cache found - loading df (with embeddings already attached) from disk...")
    df = pd.read_parquet(FEATURES_PATH)
else:
    print("No cache found - loading raw CSVs (embeddings will be computed below)...")
    df_clean = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_dataset.csv'))
    df_raw = pd.read_csv(os.path.join(DATA_DIR, 'DM1_game_dataset.csv'))
    df_text = df_raw[['BGGId', 'Description']]
    df = pd.merge(df_clean, df_text, on='BGGId', how='inner')
    df = df.dropna(subset=['Description']).reset_index(drop=True)

print(f"Dataset ready! Total games: {len(df)}")


## Generate dense embedding

In [ ]:
# Load the pre-trained Neural IR model (needed either way: to encode new queries later)
model = SentenceTransformer('all-MiniLM-L6-v2')

if USE_CACHE:
    embeddings = np.stack(df['embedding'].to_numpy()).astype('float32')
    print(f"Embeddings loaded from cache! Shape: {embeddings.shape}")
else:
    descriptions = df['Description'].tolist()

    print("Encoding descriptions into vectors...")
    embeddings = model.encode(descriptions, show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')

    print(f"Embeddings generated! Shape: {embeddings.shape}")


## Approximate k-NN with FAISS (HNSW)

In [ ]:
# Get the vector dimension size (384)
dimension = embeddings.shape[1]

if USE_CACHE:
    print("Loading FAISS index from disk...")
    index = faiss.read_index(FAISS_INDEX_PATH)
    print(f"Index loaded! Total vectors in FAISS: {index.ntotal}")
else:
    # Initialize the HNSW index
    # The '32' is the number of connections each node has in the graph
    index = faiss.IndexHNSWFlat(dimension, 32)

    print("Building the HNSW graph...")
    # Add our game embeddings to the index
    index.add(embeddings)

    print(f"Index built! Total vectors in FAISS: {index.ntotal}")

    # Persist for next time - embeddings as a column in the Parquet file
    # (row-aligned with df), FAISS index in its own native binary format.
    df_to_cache = df.copy()
    df_to_cache['embedding'] = list(embeddings)
    df_to_cache.to_parquet(FEATURES_PATH, index=False)
    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"Cached embeddings to '{FEATURES_PATH}' and the FAISS index to '{FAISS_INDEX_PATH}'.")


## Semantic Query Processing

In [5]:
def search_games_faiss(query, top_k=5):
    """
    Searches the FAISS index using semantic embeddings.
    """
    # 1. Convert the user query into a dense vector
    query_vector = model.encode([query]).astype('float32')
    
    # 2. Perform the Approximate k-NN search
    # distances are the L2 distance (lower is more similar)
    distances, indices = index.search(query_vector, top_k)
    
    # 3. Retrieve the matching games
    results = df.iloc[indices[0]].copy()
    results['faiss_distance'] = distances[0]
    
    return results[['Name', 'faiss_distance', 'Description']]

### Test the neural engine

In [6]:
user_query = "A cooperative space game involving betrayal"
results = search_games_faiss(user_query, top_k=5)

print(f"--- Semantic Results for: '{user_query}' ---\n")
for index, row in results.iterrows():
    print(f"🚀 {row['Name']} (Distance: {row['faiss_distance']:.2f})")
    print(f"   Snippet: {row['Description'][:150]}...\n")

--- Semantic Results for: 'A cooperative space game involving betrayal' ---

🚀 The Crew: The Quest for Planet Nine (Distance: 0.78)
   Snippet: cooperative tricktaking game crew quest planet player set astronaut uncertain space adventure rumor unknown planet eventful journey space extend   exc...

🚀 Contact: Signals from Outer Space (Distance: 0.85)
   Snippet: collaborative space adventure game contact follow signal know exactly come send signal want lead certain planet instead space coordinate receive stran...

🚀 Cosmic Encounter (Distance: 0.88)
   Snippet: build galactic empirein depth space alien race cosmo vie control universe alliance form shift moment moment cataclysmic battle send starship scream wa...

🚀 Star Clicker (Distance: 0.89)
   Snippet: star clicker new cooperative game christophe raimbault   author colt express   player kid try save planet evil alien parent mission far far awayldquoe...

🚀 Battlestar Galactica Collectible Card Game (Distance: 0.91)
   Snippet: colle